# L5c: K-Nearest Neighbors Approach for Classification
In this lecture, we explore the K-Nearest Neighbors (KNN) algorithm, a memory-based method for classification tasks. KNN classifies a new data point based on the majority vote of its $K$ closest neighbors in the feature space. We'll then bridge this to kernel methods from previous lectures, showing how kernel functions can replace distance metrics to enable nonlinear classification.

> __Learning Objectives:__
>
> By the end of this lecture, you will be able to:
> * __Explain the KNN classification algorithm:__ Describe how KNN uses similarity measures (distance metrics or kernels) and majority voting to classify new data points based on their nearest neighbors in the feature space.
> * __Connect distance metrics to kernel functions:__ Understand that both distance metrics and kernel functions measure similarity, and that kernels enable implicit feature space transformations for nonlinear classification.
> * __Apply kernelized KNN to nonlinear problems:__ Use kernel functions instead of explicit distance metrics to handle nonlinear decision boundaries.

Let's get started!
___

## Examples
Today, we will use the following examples to illustrate key concepts:
 
> [▶ Kernelized K-Nearest Neighbors Classification of Synthetic Data](CHEME-5820-L5c-Example-Kernerlized-KNN-Spring-2026.ipynb). In this example, we implement KNN for binary classification on a synthetic dataset, exploring how the choice of kernel function (or distance metric) affects classification accuracy. We'll compare traditional distance-based KNN with kernelized KNN to see how kernel functions implicitly transform the feature space, enabling nonlinear classification.

___

## K-Nearest Neighbor Classification
K-nearest neighbor (KNN) classification is a machine learning algorithm for classification and regression tasks. 

> __Background__ 
> 
> Developed by [Fix and Hodges in 1951](https://www.jstor.org/stable/1403797?socuuid=e7bcd649-778e-473f-8b7b-92805b5fab5f) and later refined by [Cover and Hart in 1967](https://ieeexplore.ieee.org/document/1053964) is a _supervised_ classification algorithm where an object is classified by a plurality vote of its neighbors, with the object being assigned to the class most common among its $K$ nearest neighbors, where $1 \leq K < n$. 

The algorithm finds the $K$ closest data points to a new instance in the feature space and then classifies the new instance based on the majority class among these neighbors.

> __Key assumption__ 
> 
> The key assumption of a [K-nearest neighbor classifier](https://en.wikipedia.org/wiki/K-nearest_neighbors_algorithm) is that _similar_ inputs have _similar_ labels (in classification tasks) or _similar_ outputs for K-nearest neighbor regression tasks. The critical question is how we define and measure _similarity_.

Let's look at the pseudo-code for KNN classification.

__Initialization__: Provide a reference dataset $\mathcal{D} = \{(\mathbf{x}_{i},y_{i}) \mid i = 1,2,\dots,n\}$, where $\mathbf{x}_i \in \mathbb{R}^{m}$ are $m$-dimensional feature vectors and $y_i \in \{-1,1\}$ are discrete binary labels. Specify the number of neighbors $K$, where $1 \leq K < n$. Choose a distance metric $d(\cdot,\cdot)$ (typically Euclidean distance: $d(\mathbf{x}, \mathbf{x}^*) = \|\mathbf{x} - \mathbf{x}^*\|_2$).

For each new point $\mathbf{x}^*$ to classify __do__:

1. Compute distances between $\mathbf{x}^*$ and all reference points in $\mathcal{D}$: 
    - Calculate the distance from $\mathbf{x}^*$ to all reference points in $\mathcal{D}$: $d_i \gets d(\mathbf{x}_i, \mathbf{x}^*)$ for $i = 1, 2, \dots, n$, where $d_i$ is the distance between $\mathbf{x}^*$ and the $i$-th reference point $\mathbf{x}_i$, and $d(\cdot,\cdot)$ is the chosen distance metric.

2. Find $K$ nearest neighbors between $\mathbf{x}^*$ and the reference points in $\mathcal{D}$: 
    - Identify the set $\mathcal{N}_K(\mathbf{x}^*) = \{\mathbf{x}_{i_1}, \mathbf{x}_{i_2}, \dots, \mathbf{x}_{i_K}\}$ of $K$ reference points with the __smallest distances__ to $\mathbf{x}^*$, where $i_1, i_2, \dots, i_K$ are the indices of the $K$ nearest neighbors.

3. Assign a class using a __majority vote__ of the $K$ nearest neighbors in set $\mathcal{N}_K(\mathbf{x}^*)$: 
    - Count the class labels among the $K$ nearest neighbors and assign the majority class to $\mathbf{x}^*$:
$\hat{y}^* = \arg\max_{c \in \{-1,1\}} \sum_{j=1}^{K} \mathbf{1}(y_{i_j} = c)$

where $\mathbf{1}(\cdot)$ is the __indicator function__ that returns $\mathbf{1}$ if the condition is true and $0$ otherwise. The variable $y_{i_j}$ is the label of the $j$-th nearest neighbor.

__Return__ the predicted label $\hat{y}^*$ for the new point $\mathbf{x}^*$.

### Implementation Considerations
Several practical choices affect KNN performance in real applications. The algorithm generalizes naturally to multiclass problems, but we must also select appropriate values for $K$, choose distance metrics, and consider computational constraints.

> __Multiclass adaptation__
> 
> The same KNN procedure works for any number of classes $C\ge 2$. Replace the binary label set $\{-1,1\}$ with $\{c_1,\dots,c_C\}$ and keep the majority vote step:
> $$\hat{y}^* = \arg\max_{c \in \{c_1,\dots,c_C\}} \sum_{j=1}^{K} \mathbf{1}(y_{i_j}=c)$$
> where $\mathbf{1}(\cdot)$ is the __indicator function__ that returns $1$ if the condition is true and $0$ otherwise.

The choice of $K$ affects classification performance through the bias-variance tradeoff. Small K values (e.g., $K=1$, $K=3$) create flexible decision boundaries with low bias but high variance, while large K values produce smoother boundaries with high bias but low variance.

> __Approaches for choosing K__
>
> * __Cross-validation__: Repeatedly split the training data into training/validation parts and score each candidate K on the validation part. Select the K that minimizes validation error.
> * __Heuristic starting point__: Use $K = \sqrt{n}$ where $n$ is the number of training samples as an initial guess. This data-dependent baseline provides a reasonable starting point for cross-validation search.
> * __Problem constraints__: For binary classification, choose odd K to avoid ties. For multiclass problems with $C$ classes, choose $K = mC + 1$ (where $m$ is a non-negative integer) to ensure a unique majority class.

In practice, cross-validation remains the most reliable method, while the $\sqrt{n}$ heuristic offers a computationally cheap starting point for the search. The choice of distance metric also determines how we measure similarity between data points.

> __Distance metric options__
> 
> The choice of distance metric $d(\cdot,\cdot)$ affects classification results:
> * __Euclidean__: $d(\mathbf{x},\mathbf{x}^*) = \|\mathbf{x}-\mathbf{x}^*\|_2$, the L2 norm or standard geometric distance in $m$ dimensions. In practice, the squared Euclidean distance $\|\mathbf{x}-\mathbf{x}^*\|_2^2$ is often used since it avoids the square root computation and preserves neighbor ordering (the square root is monotonic, so ranking neighbors by distance or squared distance gives the same result).
>
> * __Manhattan__: $d(\mathbf{x},\mathbf{x}^*) = \|\mathbf{x}-\mathbf{x}^*\|_1$, the L1 norm summing absolute differences across each dimension. Manhattan distance is more robust to outliers than Euclidean distance because it does not square the differences. This makes L1 preferable when the data contains outliers or when features have different scales that should not be geometrically combined. Manhattan distance also performs well in high-dimensional spaces where many features may be sparse or irrelevant.
>
> * __Cosine similarity (converted to distance)__: $d(\mathbf{x},\mathbf{x}^*) = 1 - \frac{\mathbf{x}\cdot\mathbf{x}^*}{\|\mathbf{x}\|_2\|\mathbf{x}^*\|_2}$. This measures the cosine of the angle between two vectors (an inner product normalized by magnitudes), so it focuses on orientation rather than length. Use cosine when vectors are length-normalized and direction matters more than magnitude, or in high-dimensional sparse settings where Euclidean distances are dominated by vector length instead of feature direction.

The classical KNN algorithm has a few limitations, first is its computational cost, which scales with the reference dataset size, and second is its sensitivity to the choice of distance metric, which can significantly affect classification performance depending on the data distribution and feature characteristics.

> __Challenges with Classical KNN__
> 
> * __Computational cost__: Classifying a single query requires $O(nm + n \log K)$ operations, where $n$ is the number of reference points and $m$ is the feature dimension. Distance computation (i.e., the $O(nm)$ term) dominates for large $m$.
> * __Distance concentration in high dimensions__: As feature dimension $m$ grows, and assuming i.i.d. feature components with finite variance and fixed $n$, [Beyer et al. (1999)](https://link.springer.com/chapter/10.1007/3-540-49257-7_15) showed that the relative contrast $(d_{\max} - d_{\min})/d_{\min} \to 0$, where $d_{\max}$ and $d_{\min}$ are the maximum and minimum distances from a query to all reference points. All points become approximately equidistant, so _nearest neighbor loses discriminative power_. See [the advanced notebook for a deeper dive into this phenomenon](CHEME-5820-L5c-Advanced-ConcentrationOfDistances-Spring-2026.ipynb).

The [advanced notebook](CHEME-5820-L5c-Advanced-ConcentrationOfDistances-Spring-2026.ipynb) defines the formal notation $d_{\min}^{(m)}$, $d_{\max}^{(m)}$, and $\rho_m$, and discusses rate statements for squared distances.

___

## Kernelized KNN: From Distance to Similarity

Recall from a previous lecture on kernel functions that kernels measure similarity. Both distance metrics and kernels express the intuition that similar inputs have similar labels, but in opposite directions: small distances and large kernel values both indicate similarity.

> __The Kernel Trick for KNN__
>
> A valid kernel $k(\mathbf{x}, \mathbf{z})$ implicitly computes an inner product in a transformed feature space: $k(\mathbf{x}, \mathbf{z}) = \langle \phi(\mathbf{x}), \phi(\mathbf{z}) \rangle$. In kernelized KNN, we rank neighbors by a chosen kernel similarity rather than by a raw distance. The RBF kernel is a common choice: $k(\mathbf{x}, \mathbf{z}) = \exp(-\gamma\|\mathbf{x} - \mathbf{z}\|_2^2)$. However, because it depends on squared Euclidean distance, it can still inherit distance-concentration issues in very high dimensions.

Here is the kernelized KNN algorithm using a valid kernel function instead of a distance metric.

__Initialization__: Provide a reference dataset $\mathcal{D} = \{(\mathbf{x}_{i},y_{i}) \mid i = 1,2,\dots,n\}$, where $\mathbf{x}_i \in \mathbb{R}^{m}$ are $m$-dimensional feature vectors and $y_i \in \{-1,1\}$ are discrete binary labels. Specify the number of neighbors $K$, where $1 \leq K < n$. Choose a valid kernel function $k(\cdot,\cdot)$ (e.g., the RBF kernel: $k(\mathbf{x}, \mathbf{x}^*) = \exp(-\gamma\|\mathbf{x} - \mathbf{x}^*\|_2^2)$ with $\gamma > 0$).

For each new point $\mathbf{x}^*$ to classify __do__:

1. Compute kernel similarities between $\mathbf{x}^*$ and all reference points in $\mathcal{D}$: 
    - Evaluate the kernel for $\mathbf{x}^*$ against all reference points in $\mathcal{D}$: $s_i \gets k(\mathbf{x}_i, \mathbf{x}^*)$ for $i = 1, 2, \dots, n$, where $s_i$ is the kernel similarity between $\mathbf{x}^*$ and the $i$-th reference point $\mathbf{x}_i$, and $k(\cdot,\cdot)$ is the chosen kernel function.

2. Find $K$ most similar neighbors between $\mathbf{x}^*$ and the reference points in $\mathcal{D}$: 
    - Identify the set $\mathcal{N}_K(\mathbf{x}^*) = \{\mathbf{x}_{i_1}, \mathbf{x}_{i_2}, \dots, \mathbf{x}_{i_K}\}$ of $K$ reference points with the __largest kernel values__ to $\mathbf{x}^*$, where $i_1, i_2, \dots, i_K$ are the indices of the $K$ most similar neighbors.

3. Assign a class using either an __unweighted majority vote__ or a __weighted kernel vote__ of the $K$ most similar neighbors in set $\mathcal{N}_K(\mathbf{x}^*)$: 
    - Count the class labels among the $K$ neighbors and assign the majority class to $\mathbf{x}^*$:
$\hat{y}^* = \arg\max_{c \in \{-1,1\}} \sum_{j=1}^{K} \mathbf{1}(y_{i_j} = c)$

    - __Alternative weighted scheme (kernel-weighted vote)__: weight each neighbor by its similarity to $\mathbf{x}^*$:
$\hat{y}^* = \arg\max_{c \in \{-1,1\}} \sum_{j=1}^{K} k(\mathbf{x}_{i_j}, \mathbf{x}^*)\mathbf{1}(y_{i_j} = c)$
      where $\mathbf{1}(\cdot)$ is the indicator function that returns $\mathbf{1}$ if the condition is true and $0$ otherwise. The variable $y_{i_j}$ is the label of the $j$-th most similar neighbor. Unweighted voting is simpler and treats all selected neighbors equally. Weighted voting gives more influence to the most similar neighbors, which is useful when similarity varies noticeably within the $K$-neighbor set.

__Return__ the predicted label $\hat{y}^*$ for the new point $\mathbf{x}^*$.

The only difference from classical KNN is the similarity measure: kernels compute similarities in an implicit transformed space while distances measure dissimilarity in the original feature space.

> __Example__
>
> [▶ Kernelized K-Nearest Neighbors Classification of Synthetic Data](CHEME-5820-L5c-Example-Kernerlized-KNN-Spring-2026.ipynb). This example compares distance-based KNN with kernelized KNN on synthetic overlapping circular data, demonstrating when kernels enable better classification.

___

## Summary
K-Nearest Neighbors classifies new data points by finding their $K$ closest neighbors (or most similar neighbors using kernels) in the reference dataset and assigning the majority class through voting.

> __Key Takeaways:__
>
> * **KNN uses stored training data**: KNN classifies by majority vote among the $K$ nearest neighbors, so results depend on $K$ and the similarity measure. This also means memory use and prediction time scale with the size of the reference set.
> * **Kernelized KNN ranks neighbors by similarity**: Replace distance with a kernel value and vote either unweighted or weighted by similarity. Kernel choice and its parameters shape the neighborhood and can capture nonlinear structure.
> * **Multiclass extension keeps the same vote rule**: Replace the binary label set with $C$ classes and select the class with the most votes. Tie handling and $K$ selection matter more as the number of classes grows.

KNN remains widely used in practice. Kernelized KNN handles nonlinear patterns and connects to kernel methods such as Support Vector Machines (later lectures).

___